# Exercícios Aula 05: Rastreamento, Detecção e Reconhecimento Facial, Pose Estimation

Este notebook contém **5 exercícios práticos** para reforçar os principais conceitos apresentados nos notebooks da Aula 05 (`05_01` a `05_05`):

1. Persistência de IDs no rastreamento com YOLO + ByteTrack
2. Confirmação de rastros com a biblioteca `trackers` da Roboflow
3. Seleção da pessoa principal em uma foto de grupo
4. Verificação de identidade 1:1 com embeddings faciais
5. Lendo landmarks de pose para detectar a posição das mãos

Complete os blocos marcados com `# TODO` em cada exercício. Use os notebooks `05_01` a `05_05` como referência sempre que precisar relembrar a sintaxe de alguma função.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93
    !pip install opencv-contrib-python==5.0.0.93
    !pip install ultralytics
    !pip install moviepy==2.2.1
    !pip install trackers
    !pip install supervision
    !pip install insightface
    !pip install onnxruntime
    !pip install mediapipe==1.0.1
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Vamos combinar as bibliotecas usadas em todos os notebooks da Aula 05: `ultralytics` e `trackers` (rastreamento), `insightface` (detecção e reconhecimento de faces) e `mediapipe` (pose estimation).

In [ ]:
from ultralytics import YOLO
from trackers import ByteTrackTracker
from insightface.app import FaceAnalysis
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision

from collections import defaultdict
from pathlib import Path
from urllib.request import urlretrieve

import cv2
import supervision as sv
import numpy as np
import matplotlib.pyplot as plt
# Indica ao notebook to render figures in-page.
%matplotlib inline

print("Bibliotecas carregadas!")

## Preparação: Modelo YOLO e Vídeo de Trânsito

Antes dos Exercícios 1 e 2, vamos carregar o `YOLO('modelos/yolo26n.pt')` (o mesmo modelo de `05_01` e `05_02`) e apontar para o vídeo de trânsito. Essa célula é apenas infraestrutura, não faz parte dos exercícios.

In [ ]:
detect_model = YOLO('modelos/yolo26n.pt')

video_path = 'imagens/05/2099406-hd_1920_1080_30fps.mp4'

print("Modelo e vídeo prontos!")

## Exercício 1: Persistência de IDs no Rastreamento com YOLO + ByteTrack

**Conceito reforçado:** `persist=True` diz à `ultralytics` para manter o estado do `ByteTrack` entre uma chamada de `track()` e outra, associando as detecções do quadro atual com os rastros dos quadros anteriores. Sem ele, cada quadro recomeça o rastreamento do zero e todo objeto ganha um ID novo, tornando impossível contar objetos únicos ao longo do vídeo.

1. Abra o vídeo com `cv2.VideoCapture(video_path)`.
2. Processe os primeiros 90 quadros do vídeo chamando `detect_model.track(quadro, persist=True, tracker='bytetrack.yaml', verbose=False)` em cada um, guardando os IDs únicos de rastreamento por classe em um `defaultdict(set)`, como em `05_01`.
3. Imprima quantos objetos únicos de cada classe foram rastreados nesses 90 quadros.
4. Refaça a mesma contagem, desta vez chamando `track()` sem `persist=True` a cada quadro. O número de IDs únicos aumentou, diminuiu ou ficou igual? Por quê? Escreva sua conclusão em um comentário.

In [ ]:
# 1. Abra o vídeo com cv2.VideoCapture(video_path)


In [ ]:
# 2. Processe os primeiros 90 quadros com track(persist=True, tracker='bytetrack.yaml'), contando IDs únicos por classe


In [ ]:
# 3. Imprima a contagem de objetos únicos por classe


In [ ]:
# 4. Refaça a contagem sem persist=True e escreva sua conclusão:


## Exercício 2: Confirmação de Rastros com a Biblioteca `trackers`

**Conceito reforçado:** diferente do `ByteTrack` embutido na `ultralytics`, que confirma um ID já no primeiro quadro, o `ByteTrackTracker` da Roboflow só confirma um `tracker_id` depois que o objeto aparece em `minimum_consecutive_frames` quadros consecutivos (2, por padrão). Detecções ainda não confirmadas saem com `tracker_id == -1`.

1. Abra o vídeo com `cv2.VideoCapture(video_path)` e capture o primeiro quadro.
2. Rode `detect_model.predict()` nesse quadro e converta o resultado para `sv.Detections` com `sv.Detections.from_ultralytics()`.
3. Crie um `ByteTrackTracker()` novo e chame `update()` três vezes seguidas com essas mesmas detecções, imprimindo `tracker_id` a cada chamada e contando quantas detecções ainda estão em `-1`.
4. A partir de que chamada todos os IDs ficaram confirmados? Isso bate com o `minimum_consecutive_frames` padrão do rastreador? Escreva sua conclusão em um comentário.

In [ ]:
# 1. Abra o vídeo e capture o primeiro quadro


In [ ]:
# 2. Rode predict() no quadro e converta para sv.Detections


In [ ]:
# 3. Crie um ByteTrackTracker() e chame update() três vezes, imprimindo tracker_id e contando os -1


In [ ]:
# 4. A partir de que chamada todos os IDs ficaram confirmados? Escreva sua conclusão:


## Preparação: InsightFace (Detecção)

Antes do Exercício 3, vamos carregar o `FaceAnalysis` só com o módulo de detecção, como na Seção 1.1 de `05_03`, e reaproveitar a função `desenhar_rostos()` desse mesmo notebook.

In [ ]:
app_deteccao = FaceAnalysis(name='buffalo_l', allowed_modules=['detection'])
app_deteccao.prepare(ctx_id=0, det_size=(640, 640))

def desenhar_rostos(img: np.ndarray, faces: list, cor: tuple = (0, 255, 0), espessura: int = 2) -> np.ndarray:
    """ Desenha a caixa delimitadora, a confiança e os pontos de referência de cada rosto """
    img = img.copy()

    for face in faces:
        x1, y1, x2, y2 = face.bbox.round().astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), cor, espessura)
        cv2.putText(
            img,
            f"{face.det_score:.2f}",
            (x1, max(y1 - 8, 0)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            cor,
            espessura
        )
        for x, y in face.kps.round().astype(int):
            cv2.circle(img, (x, y), 2, cor, -1)

    return img

print("Modelo e função de apresentação prontos!")

## Exercício 3: Seleção da Pessoa Principal em uma Foto de Grupo

**Conceito reforçado:** a `insightface` não indica diretamente quem é a "pessoa principal" de uma foto, mas o rosto de maior área da caixa delimitadora costuma ser o de quem está mais perto da câmera, a mesma heurística usada em `05_03` para selecionar a pessoa principal de uma selfie.

1. Carregue a imagem `imagens/05/selecao-brasileira.webp` e detecte os rostos com `app_deteccao.get()`.
2. Defina uma função `area_rosto()` que calcula a área da caixa delimitadora de um rosto e use-a para encontrar o rosto de maior área entre todos os detectados.
3. Desenhe todos os rostos em cinza fino (`desenhar_rostos(..., cor=(160, 160, 160), espessura=1)`) e destaque o rosto de maior área em uma cor diferente, mais grossa.
4. Esse rosto está necessariamente mais perto da câmera, ou pode haver outro motivo para ele aparecer maior na imagem (zoom, lente, posição do jogador na foto)? Escreva sua hipótese em um comentário.

In [ ]:
# 1. Carregue a imagem 'imagens/05/selecao-brasileira.webp' e detecte os rostos


In [ ]:
# 2. Defina area_rosto() e encontre o rosto de maior área


In [ ]:
# 3. Desenhe todos os rostos em cinza fino e destaque o de maior área


In [ ]:
# 4. Escreva sua hipótese sobre por que esse rosto aparece maior:


## Preparação: InsightFace (Detecção + Reconhecimento)

Antes do Exercício 4, vamos carregar o `FaceAnalysis` com os módulos de detecção e reconhecimento, como na Seção 1.1 de `05_04`.

In [ ]:
app_reconhecimento = FaceAnalysis(name='buffalo_l', allowed_modules=['detection', 'recognition'])
app_reconhecimento.prepare(ctx_id=0, det_size=(640, 640))

print("Modelo de reconhecimento pronto!")

## Exercício 4: Verificação de Identidade 1:1 com Embeddings Faciais

**Conceito reforçado:** na verificação 1:1, comparamos o embedding de uma selfie de consulta com o de uma única foto de referência, aplicando um limiar de similaridade para decidir "verificado" ou "não verificado" — diferente da busca 1:N da Seção 3 de `05_04`, que compara com um banco inteiro.

1. Defina uma função `area_rosto()` (como em `05_04`) e uma função `extrair_embedding(image_path)` que detecta o rosto principal de uma imagem e devolve seu `normed_embedding`.
2. Extraia o embedding da consulta `imagens/05/banco/Donald_Rumsfeld/Donald_Rumsfeld_0005.jpg`.
3. Extraia também o embedding da foto `imagens/05/banco/Donald_Rumsfeld/Donald_Rumsfeld_0001.jpg` (identidade correta) e da foto `imagens/05/banco/Colin_Powell/Colin_Powell_0001.jpg` (identidade errada).
4. Calcule a similaridade de cosseno da consulta com cada uma das duas referências, aplique um limiar de verificação (por exemplo, `0.45`) e imprima se cada comparação foi "verificada" ou "não verificada". As duas decisões saíram como esperado? Escreva sua conclusão em um comentário.

In [ ]:
# 1. Defina area_rosto() e extrair_embedding(image_path)


In [ ]:
# 2. Extraia o embedding da consulta 'Donald_Rumsfeld_0005.jpg'


In [ ]:
# 3. Extraia os embeddings de referência (identidade correta e identidade errada)


In [ ]:
# 4. Calcule as similaridades, aplique o limiar e escreva sua conclusão:


## Preparação: PoseLandmarker do MediaPipe

Antes do Exercício 5, vamos baixar o modelo `pose_landmarker_lite.task` (reaproveitando `baixar_arquivo_modelo()` de `05_05`), carregar o `PoseLandmarker` em modo `IMAGE`, e reaproveitar a função `desenhar_esqueleto()` desse mesmo notebook.

In [ ]:
def baixar_arquivo_modelo(url: str, nome_arquivo: str) -> Path:
    # Pasta para armazenar os modelos
    MODELOS_DIR = Path('modelos')
    MODELOS_DIR.mkdir(exist_ok=True)

    path_modelo = MODELOS_DIR / nome_arquivo

    if not path_modelo.exists():
        print("Baixando arquivo...")
        urlretrieve(url, path_modelo)
        print("Download concluído!")
    else:
        print("Arquivo já existe.")

    return path_modelo

url = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
)

modelo_path = baixar_arquivo_modelo(url, 'pose_landmarker_lite.task')

options = mp_vision.PoseLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=str(modelo_path)),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_poses=1,
)
landmarker = mp_vision.PoseLandmarker.create_from_options(options)

PoseLandmark = mp_vision.PoseLandmark
CONEXOES = mp_vision.PoseLandmarksConnections.POSE_LANDMARKS

def desenhar_esqueleto(img: np.ndarray, landmarks: list, cor: tuple = (0, 220, 0)) -> np.ndarray:
    """ Desenha o esqueleto (pontos + conexões) sobre a imagem """
    img = img.copy()
    altura, largura = img.shape[:2]

    pontos = [(int(lm.x * largura), int(lm.y * altura)) for lm in landmarks]

    for conexao in CONEXOES:
        cv2.line(img, pontos[conexao.start], pontos[conexao.end], cor, 3, cv2.LINE_AA)
    for x, y in pontos:
        cv2.circle(img, (x, y), 5, cor, -1, cv2.LINE_AA)

    return img

print("Modelo e função de apresentação prontos!")

## Exercício 5: Lendo Landmarks de Pose para Detectar a Posição das Mãos

**Conceito reforçado:** `pose_landmarks` traz coordenadas normalizadas (`x`, `y` entre 0 e 1), com `y` crescendo de cima para baixo na imagem. Comparar a coordenada `y` de landmarks diferentes já é o bastante para responder perguntas simples de postura, sem precisar da complexidade do `pose_world_landmarks` usado para medir giros na Seção 3 de `05_05`.

1. Capture o primeiro quadro do vídeo `imagens/05/7342766-uhd_2160_3840_25fps.mp4` (`cv2.VideoCapture`, convertendo para RGB) e detecte os pontos do corpo com `landmarker.detect()`, passando um `mp.Image`.
2. A partir de `resultado.pose_landmarks[0]`, pegue as coordenadas normalizadas de `PoseLandmark.NOSE`, `PoseLandmark.LEFT_WRIST` e `PoseLandmark.RIGHT_WRIST`.
3. Compare a coordenada `y` de cada pulso com a do nariz para decidir se aquela mão está acima ou abaixo da cabeça, e imprima o resultado para as duas mãos.
4. Desenhe o esqueleto completo sobre o quadro com `desenhar_esqueleto()` para conferir visualmente se sua conclusão da etapa 3 bate com a pose da pessoa.

In [ ]:
# 1. Capture o primeiro quadro do vídeo e detecte os pontos do corpo


In [ ]:
# 2. Pegue as coordenadas normalizadas de NOSE, LEFT_WRIST e RIGHT_WRIST


In [ ]:
# 3. Compare o y de cada pulso com o do nariz e imprima se a mão está acima ou abaixo da cabeça


In [ ]:
# 4. Desenhe o esqueleto sobre o quadro com desenhar_esqueleto()
